In [1]:
!uv pip install datasets pandas scikit-learn transformers

Using Python 3.14.2 environment at: /home/introlix/Desktop/huemeet/backend/.venv
Resolved 45 packages in 539ms                                        
Uninstalled 1 package in 7ms
Installed 34 packages in 470ms                              
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + attrs==25.4.0
 + datasets==4.5.0
 + dill==0.4.0
 + frozenlist==1.8.0
 - fsspec==2026.1.0
 + fsspec==2025.10.0
 + hf-xet==1.2.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.4.1
 + joblib==1.5.3
 + multidict==6.7.1
 + multiprocess==0.70.18
 + numpy==2.4.2
 + packaging==26.0
 + pandas==3.0.0
 + propcache==0.4.1
 + pyarrow==23.0.0
 + python-dateutil==2.9.0.post0
 + pyyaml==6.0.3
 + safetensors==0.7.0
 + scikit-learn==1.8.0
 + scipy==1.17.0
 + shellingham==1.5.4
 + six==1.17.0
 + threadpoolctl==3.6.0
 + tokenizers==0.22.2
 + tqdm==4.67.3
 + transformers==5.1.0
 + typer-slim==0.21.1
 + xxhash==3.6.0
 + yarl==1.22.0


## Import and load dataset

In [1]:
# Imports 
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TextClassificationPipeline
from datasets import load_dataset

/home/introlix/Desktop/huemeet/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Models
# 1. https://huggingface.co/unitary/unbiased-toxic-roberta
# 2. https://huggingface.co/facebook/roberta-hate-speech-dynabench-r4-target
# 3. https://huggingface.co/martin-ha/toxic-comment-model

In [2]:
# Configurations
MODEL_NAME = "unitary/unbiased-toxic-roberta"

In [3]:
# Datasets
# 1. https://huggingface.co/datasets/textdetox/multilingual_toxicity_dataset/

In [4]:
ds = load_dataset("textdetox/multilingual_toxicity_dataset")

In [5]:
df = ds['en'].to_pandas()

In [ ]:
df.sample(5)

##  Load Model

In [6]:
# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

pipeline =  TextClassificationPipeline(model=model, tokenizer=tokenizer, return_all_scores=True)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 552.03it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: unitary/unbiased-toxic-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# a test sentence
text = "You are wasting my time."
result = pipeline(text)
print(result)

0.005196026526391506


## Evaluation

### Eval on HF dataset

In [ ]:
# eval for 
y_true = []
y_pred = []
y_prob = []

for item in ds['en']:
    text = item['text']
    label = item['toxic']
    
    result = pipeline(text)
    # prob = result[0]['score']  # probability of being toxic
    # pred_label = 1 if prob >= 0.5 else 0  # thresholding at 0.5
    pred_label = 0 if result[0]['label'] == 'non-toxic' else 1 
    
    y_true.append(label)
    y_pred.append(pred_label)
    # y_prob.append(prob)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 Score :", f1_score(y_true, y_pred))

### Eval On My Dataset

In [11]:
# Loading Data
df = pd.read_csv("../data/toxic.csv")

In [13]:
df.head()

,text,label
0,You are an idiot and you don't know anything,toxic
1,He is a fucking person,toxic
2,You are the worst programmer I have ever seen,toxic
3,Go kill yourself,toxic
4,Shut your mouth you stupid fool,toxic


In [42]:
# eval for 
y_true = []
y_pred = []
y_prob = []

for _, row in df.iterrows():
    text = row['text']
    label = 1 if row['label'] == 'toxic' else 0
    
    result = pipeline(text)
    # prob = result[0]['score']  # probability of being toxic
    # pred_label = 1 if prob >= 0.5 else 0  # thresholding at 0.5
    pred_label = 0 if result[0]['label'] == 'non-toxic' else 1 
    
    y_true.append(label)
    y_pred.append(pred_label)
    # y_prob.append(prob)

In [43]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 Score :", f1_score(y_true, y_pred))

Accuracy : 0.8695652173913043
Precision: 0.9705882352941176
Recall   : 0.66
F1 Score : 0.7857142857142857


## Final Results of All Model

#### On textdetox/multilingual_toxicity_dataset
1. Model: unitary/unbiased-toxic-roberta
- Accuracy : 0.781
- Precision: 0.7042744984006979
- Recall   : 0.9688
- F1 Score : 0.8156255261828591

2. Model: facebook/roberta-hate-speech-dynabench-r4-target
- Accuracy : 0.7792
- Precision: 0.8166969147005445
- Recall   : 0.72
- F1 Score : 0.7653061224489796

3. Model: martin-ha/toxic-comment-model
- Accuracy : 0.846
- Precision: 0.9417773237997957
- Recall   : 0.7376
- F1 Score : 0.8272768057424854

#### On toxic.csv dataset
1. Model: unitary/unbiased-toxic-roberta
- Accuracy : 0.9710144927536232
- Precision: 0.9791666666666666
- Recall   : 0.94
- F1 Score : 0.9591836734693877

2. Model: facebook/roberta-hate-speech-dynabench-r4-target
- Totally 

3. Model: martin-ha/toxic-comment-model
- Accuracy : 0.8695652173913043
- Precision: 0.9705882352941176
- Recall   : 0.66
- F1 Score : 0.7857142857142857